## Importing libraries and load environment variables


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

print("API LOADED SUCCESSFULLY")

API LOADED SUCCESSFULLY


## Initialize the GROQ LLM

In [14]:
# import time
# from langchain_groq import ChatGroq

# model=ChatGroq(
#     model="openai/gpt-oss-20b",
#     temperature=0, max_retries=2
# )
# response=model.invoke("Explain RAG in One Sentence.")
# print(response.content)

# for chunk in model.stream("Explain RAG in 50 sentence."):
#     print(chunk.content, end="", flush=True)
#     time.sleep(0.02)


from langchain.chat_models import init_chat_model
model =init_chat_model("openai/gpt-oss-20b", model_provider="groq")


for chunk in model.stream("Explain RAG in 50 sentence."):
    print(chunk.content, end="", flush=True)

Retrieval-Augmented Generation (RAG) is a framework that enhances language model responses by incorporating external knowledge retrieved from a document collection.  
It was introduced to address the limitations of purely generative models that rely solely on their internal parameters for knowledge.  
RAG works by first querying a retriever to fetch a set of relevant documents based on the input prompt.  
The retrieved documents are then fed into a generator, which conditions its output on both the prompt and the external text.  
This two-step process allows the model to access up-to-date or domain‑specific information that may not be encoded in its weights.  
The retriever component can be a dense embedding model, a BM25 index, or any other search mechanism.  
In practice, dense retrievers such as DPR or Sentence‑BERT are popular because they capture semantic similarity.  
BM25, a bag‑of‑words ranking algorithm, remains useful for its speed and simplicity.  
The generator is typically

## Load the DOCX document


In [40]:
from langchain_community.document_loaders import Docx2txtLoader

file_path="../data/company_info.docx"

loader=Docx2txtLoader(file_path)

documents=loader.load()

print("Number of documents:", len(documents))
print(documents[0].page_content[:1000])

Number of documents: 1
Project: Banking Application

The Banking Application is a backend-based financial application developed to manage customer accounts and banking operations.

Technology Stack

The backend is developed using Java and Spring Boot.

The application uses PostgreSQL as the primary database.

The application provides REST APIs for account management and banking operations.

The project uses Spring Data JPA for database interaction and Hibernate as the ORM framework.

Maven is used for dependency management and project build.

Git is used for source code management.

Account Management

The application allows customers to:

Create a bank account

View account details

Update customer information

Check account balance

Close an account

Search accounts using customer or account information

Each bank account has a unique account number associated with a customer.

Transaction Management

The application supports basic banking transactions such as:

Money deposit

Money 

## Split the document into chunks

In [46]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_Splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=30
)

chunks=text_Splitter.split_documents(documents)

print("Number of chunks: ", len(chunks))
for i, chunk in enumerate(chunks[:3]):
    print(f"\nchunk {i+1}:")
    print(chunk.page_content[:300])

Number of chunks:  30

chunk 1:
Project: Banking Application

The Banking Application is a backend-based financial application developed to manage customer accounts and banking operations.

Technology Stack

chunk 2:
Technology Stack

The backend is developed using Java and Spring Boot.

The application uses PostgreSQL as the primary database.

chunk 3:
The application provides REST APIs for account management and banking operations.

The project uses Spring Data JPA for database interaction and Hibernate as the ORM framework.


## Configure Google embeddings


In [47]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings=GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)
test_vector=embeddings.embed_query("What is Spring Boot?")
print("Embedding dimensions: ", len(test_vector))

Embedding dimensions:  3072


## Create the FAISS vectore store

In [48]:
from langchain_community.vectorstores import FAISS

vectorstore=FAISS.from_documents(documents=chunks, embedding=embeddings)

print("FAISS vector store created successfully")

FAISS vector store created successfully


## Save the vector store locally

In [49]:
vectorstore.save_local("../vectorestore/company_faiss")
print("FAISS vectore store saved")

FAISS vectore store saved


## Test document retrieval

In [50]:
retriever=vectorstore.as_retriever(
    search_kwargs={"k":2}
)

query="Which technology is used in the backend?"

retriever_docs=retriever.invoke(query)

for i, doc in enumerate(retriever_docs):
    print(f"\n Document {i+1}")
    print(doc.page_content)


 Document 1
Technology Stack

The backend is developed using Java and Spring Boot.

The application uses PostgreSQL as the primary database.

 Document 2
Project: Banking Application

The Banking Application is a backend-based financial application developed to manage customer accounts and banking operations.

Technology Stack


## Create the RAG search tool

In [ ]:
from langchain_community import tools
@tool
def search_document(query: str)->str:
    """Search the company DOCX docummeys for relevent information. Use this tool
    when the user asks about information that may be present in the uploaded document."""

    docs=retriever.invoke(query)

    if not docs:
        return "No relevant information found in the document."

    results=[]

    for i, doc in enumerate(docs):
        results.append(
            f"Document chunk {i+1}: \n{doc.page_content}"
        )

    return "\n\n".join(results)

NameError: name 'tool' is not defined

## Test the tool

In [ ]:
result=search_document.invo